# Module 2 本地练习：第一个手写数字分类器

这是根据你提供的 Module 2 讲义编写的学习练习，不是官方实验文件。对应 Image Classification Part 1 / Part 2，使用 MNIST 数字图片；后面的 EMNIST 字母作业是另一项任务。

目标：走通图片 → Dataset → DataLoader → 自定义模型 → 训练 → 测试 → 查看预测。

数据和新语法示例已提供，你主要填写两个地方：模型结构和每批数据的训练步骤。遇到 `NotImplementedError` 表示到了待填写位置，不是环境故障。填写后删掉对应的 `raise` 行。

按三个大块学习即可，不用每个单元都交一次答案。先完成数据与模型这一块，再一起检查。


## 开始前：选择内核和配置环境

用 VS Code 打开此文件，右上角选择你的学习环境 `.venv`。先运行下面的诊断单元。

本次检查时，你的学习环境安装了 CPU 版 PyTorch，尚未安装 torchvision。你有 RTX 5070 Laptop GPU，可以选择 CUDA 版。

**以下命令需要你自己执行；本文件没有自动安装单元。** 若使用原学习环境，在这个 notebook 中临时新建一个代码单元，运行：

```python
%pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu128
```

这是官方列出的 CUDA 12.8 配套版本，会替换当前内核环境里的 CPU 版 torch。不是要求课程必须使用这个版本，而是提供一个已核对的安装组合。安装会下载较大的文件。安装结束后**重启内核**，再从上向下运行。

如果缺少 matplotlib，在临时单元运行 `%pip install matplotlib`，随后重启内核。

`%pip` 在当前 notebook 的内核环境中安装包，避免装进另一个 Python。只看说明或运行诊断不会改变环境。若暂时继续使用 CPU，也能完成此练习；需要安装与你的 torch 配套的 torchvision。

版本依据：[PyTorch 官方安装组合](https://pytorch.org/get-started/previous-versions/)。数据接口：[MNIST 官方文档](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html)。


In [ ]:
import sys
import importlib.util
import torch

print('当前 Python：', sys.executable)
print('PyTorch：', torch.__version__)
print('CUDA 运行时：', torch.version.cuda)
print('GPU 可用：', torch.cuda.is_available())
print('torchvision 已安装：', importlib.util.find_spec('torchvision') is not None)
if torch.cuda.is_available():
    print('显卡：', torch.cuda.get_device_name(0))


## 第一块：准备数据，理解图片和标签

这里沿用讲义的配置。`ToTensor()` 将 MNIST 图片转成浮点张量，并把像素从 0～255 缩放到 0～1；`Normalize()` 再按讲义给定的均值、标准差进行标准化。

`Dataset` 按索引提供一张图片和它的正确标签；`DataLoader` 把它们组合成批。标签是正确数字 0～9，不是模型猜的数字。

首次运行数据下载单元需要联网，会在 **当前工作目录的 data 文件夹**保存数据，后续可以复用。本练习不需要 helper_utils.py。


In [ ]:
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from pathlib import Path

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('本次计算设备：', device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


In [ ]:
data_dir = Path.cwd() / 'data'
print('数据保存位置：', data_dir)
train_dataset = torchvision.datasets.MNIST(
    root=str(data_dir), train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root=str(data_dir), train=False, download=True, transform=transform
)

# num_workers=0：先使用简单的加载方式，方便在 Windows notebook 中学习。
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False, num_workers=0)
print('训练图片数：', len(train_dataset), '测试图片数：', len(test_dataset))


In [ ]:
# iter 创建迭代器，next 取出一批；这一步不会训练模型。
images, labels = next(iter(train_loader))
print('图片形状：', images.shape)  # [批大小, 通道数, 高, 宽]
print('标签形状：', labels.shape, '标签类型：', labels.dtype)
print('前八个正确标签：', labels[:8])

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    # 将标准化后的图片还原到 0～1，方便显示。
    ax.imshow(images[i, 0] * 0.3081 + 0.1307, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'label={labels[i].item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 第二块：自己补全模型

讲义结构：`Flatten → Linear(784, 128) → ReLU → Linear(128, 10)`。

- `class MNISTClassifier(nn.Module)`：定义一种自己的模型，继承 PyTorch 模型的功能。
- `__init__`：创建模型时设置层；`super().__init__()` 先完成父类的初始化。
- `self.xxx`：把层保存在这个模型对象里，训练时 PyTorch 才能找到这些参数。
- `forward`：规定输入依次经过哪些层；调用 `model(images)` 时会执行它。

每张图片有 1×28×28=784 个数。`Flatten()` 默认保留第 0 维的批大小，把后面的维度展平。

请补全 `self.layers`。模型最后输出的是每个数字的 **10 个分数（logits）**，不加 ReLU 或 Softmax；后面的 CrossEntropyLoss 直接接收这些分数。


In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        # TODO：用 nn.Sequential 创建上面要求的三层，赋给 self.layers。
        raise NotImplementedError('请补全 self.layers，并删除这行 raise')

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

model = MNISTClassifier().to(device)
print(model)


In [ ]:
# 只检查接口：一批图片应产生 [批大小, 10] 的分数。
with torch.no_grad():
    scores = model(images.to(device))
assert tuple(scores.shape) == (images.shape[0], 10), scores.shape
parameter_count = sum(p.numel() for p in model.parameters())
print('模型输出形状：', scores.shape)
print('可训练参数数量：', parameter_count)
assert parameter_count == 101770, '请核对层的输入、输出大小和偏置设置'
print('模型结构检查通过。此时参数仍然没有经过训练。')


## 第三块：训练、测试和查看预测

回归时我们用了 MSELoss；这次标签是类别编号，用 `CrossEntropyLoss()`。标签形状保持 `[批大小]`，类型为 `torch.int64`，不要像之前回归那样加成一列。

这次优化器沿用讲义的 Adam。你熟悉的训练顺序保持不变：清空梯度 → 前向预测 → 计算损失 → 反向计算梯度 → 更新参数。

下面只需在 TODO 区域写出这五步。使用变量名 `outputs` 和 `loss`，后面的统计代码会读取它们。


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # TODO：在这里写出五步训练代码，预测命名为 outputs，损失命名为 loss。
        raise NotImplementedError('请填写五步训练代码，并删除这行 raise')

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        predicted = outputs.detach().argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

    return total_loss / total, correct / total


下面的评估代码已提供。先看懂它与训练的区别即可：

- `model.eval()` 切换到评估模式；`torch.no_grad()` 关闭梯度记录。评估没有 backward 和 step。
- `argmax(dim=1)`：对每张图片的十个分数取最大分数所在的类别编号；不是取最大分数本身。
- `(predicted == labels).sum().item()`：统计本批预测正确的图片数。
- 损失乘本批样本数再累计，最后除以总样本数，避免最后一个小批次的权重算错。

测试集用来做最后检查。如果之后反复用它选择网络结构，它就承担了验证集的角色；正式比较模型时应另外划分验证集。


In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            predicted = outputs.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


In [ ]:
# 第一次先跑 1 轮，确认流程正常；之后可以改为讲义的 10 轮。
# 再运行本单元会继续训练当前模型。若要从头训练，重新运行模型和优化器单元。
epochs = 1
history = []
for epoch in range(epochs):
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    history.append((train_loss, train_accuracy))
    print(f'Epoch {epoch + 1}/{epochs} | loss={train_loss:.4f} | '
          f'train accuracy={train_accuracy:.2%}')

test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)
print(f'Test loss={test_loss:.4f} | test accuracy={test_accuracy:.2%}')


In [ ]:
model.eval()
test_images, test_labels = next(iter(test_loader))
with torch.no_grad():
    predictions = model(test_images[:12].to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(test_images[i, 0] * 0.3081 + 0.1307, cmap='gray', vmin=0, vmax=1)
    good = predictions[i].item() == test_labels[i].item()
    ax.set_title(f'true={test_labels[i].item()}, pred={predictions[i].item()}',
                 color='green' if good else 'red')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 做到哪里发给我

先完成数据部分和模型 TODO，把模型代码、图片形状、标签形状、模型输出形状一起发给我。之后再集中检查训练与评估，不需要逐单元回答测验。

完成训练后发训练日志、测试准确率，以及你发现的一张预测错误的图片（前十二张都正确也正常，可以看更多图片）。我们再结合结果解释错在哪里。

本练习的意义是把课程的整个工作流亲手接起来；不规定一次运行必须达到某个准确率。随机初始化、训练轮数和环境都会影响结果。
